## Agent란?
- LLM이 스스로 판단해서 어떤 행동(툴 사용 포함)을 할 지 결정하느 실행주체를 의미합니다.

In [1]:
from dotenv import load_dotenv
from langchain_openai.chat_models.base import ChatOpenAI
import os

load_dotenv()
print(os.environ.get('OPENAI_API_KEY')[:20])

sk-proj-gmk90JKmSW4I


## agent를 만들어야 하는 이유 

In [2]:
# 5,243.38
prompt = "cost of $355.39 + $924.87 + $721.2 + $1940.29 + $573.63 + $65.72 + $35.00 + $522.00 + $76.16 + $29.12"

In [24]:
chat = ChatOpenAI(temperature=0.1)

In [25]:
# 5,243.38
result = chat.invoke(prompt)

In [26]:
# 5,243.38
result.content

'The total cost is $4,363.38.'

In [27]:
chat = ChatOpenAI(model="gpt-4o",temperature=0.1)

In [28]:
# 5,243.38
result = chat.invoke(prompt)

In [29]:
# 5,243.38
print(result.content)

To find the total cost, you need to add all the amounts together:

$355.39 + $924.87 + $721.20 + $1940.29 + $573.63 + $65.72 + $35.00 + $522.00 + $76.16 + $29.12 = $5243.38

The total cost is $5243.38.


In [30]:
"""
    프롬프트의 정답
    $4,363.38.

    계산기에서 직접 계산하기 ↓↓↓
    $5,243.38

    llm의 계산 착오
    ※이유※
    LLM은 산술 연산을 수행하지 않습니다. 이런 계산은 AI보다 계산기가 더 잘합니다.
    LLM은 text를 생성해내는 모델입니다. 문장의 시퀀스의 다음 token이 무엇인지 통계적으로 추측합니다.
    이러한 LLm의 오류를 잡기 위해서는 agent를 제공해주어야 합니다.
    그리고 agent를 위한 tool(툴)을 만들고, agent가 tool을 선택해서 실행하는 것입니다.
"""
None

## Agent 생성

In [3]:
# create_agent: 함수로 통일
from langchain.agents import create_agent
from langchain.tools import tool

In [32]:
@tool
def plus(num1: float, num2: float) -> float:
    """
        Adds two numbers and return the result.
    """
    return num1 + num2

"""
    json{
        "name": plus,
        "description": "Adds two numbers and return the result."
        "parameters":{
            "num1": "float",
            "num2": "float"
        }
    }
"""
None

In [33]:
agent = create_agent(
    model="gpt-3.5-turbo",
    tools=[plus],
    system_prompt="You are a helpful assistant"
)

In [39]:
result = agent.invoke({
    "messages":[
        {
            "role": "user",
            "content": prompt
        }
    ]
})

In [40]:
# $5,265.16
result

{'messages': [HumanMessage(content='cost of $355.39 + $924.87 + $721.2 + $1940.29 + $573.63 + $65.72 + $35.00 + $522.00 + $76.16 + $29.12', additional_kwargs={}, response_metadata={}, id='1e5ca9e4-b1dc-48fb-a30d-a71a819952c2'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 131, 'prompt_tokens': 109, 'total_tokens': 240, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-Dh4UDgVvYNeFGylQYbiKPjJvGYse8', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e3e10-5dc9-77f3-8b65-9d8ac8cd6659-0', tool_calls=[{'name': 'plus', 'args': {'num1': 355.39, 'num2': 924.87}, 'id': 'call_UD7FUcyrvhPdMx6HxHirvIh8', 'type': 'tool_call'},

In [41]:
for message in result["messages"]:
    if message.__class__.__name__ == "AIMessage" and message.tool_calls:
        for i in message.tool_calls:
            print(i)

{'name': 'plus', 'args': {'num1': 355.39, 'num2': 924.87}, 'id': 'call_UD7FUcyrvhPdMx6HxHirvIh8', 'type': 'tool_call'}
{'name': 'plus', 'args': {'num1': 721.2, 'num2': 1940.29}, 'id': 'call_ndGNjfNuKrw866tqtqcQgtMH', 'type': 'tool_call'}
{'name': 'plus', 'args': {'num1': 573.63, 'num2': 65.72}, 'id': 'call_2ZY4rj6VQASuVDfdx1JsOISZ', 'type': 'tool_call'}
{'name': 'plus', 'args': {'num1': 35.0, 'num2': 522.0}, 'id': 'call_UuUaQlXzRad1y6bR9Kv0RwQb', 'type': 'tool_call'}
{'name': 'plus', 'args': {'num1': 76.16, 'num2': 29.12}, 'id': 'call_suSkRxSKrjYqsnLCBGyeuMPf', 'type': 'tool_call'}


In [4]:
@tool
def total_sum(numbers: list[float]) -> float:
    """
        Adds a list of numbers and returns the total sum.
        Use this tool when you need to calculate the total of multiple numbers.
        Input should be a string representation of a list.
        Example: "[1, 2, 3]"
    """
    return sum(numbers)

In [5]:
agent = create_agent(
    model="gpt-3.5-turbo",
    tools=[total_sum],
    system_prompt="You are a helpful assistant"
)

In [6]:
result = agent.invoke({
    "messages":[
        {
            "role": "user",
            "content": prompt
        }
    ]
})

In [7]:
# $5243.38
result["messages"][-1].content

'The total sum of $355.39 + $924.87 + $721.2 + $1940.29 + $573.63 + $65.72 + $35.00 + $522.00 + $76.16 + $29.12 is $5243.38.'

In [48]:
for message in result["messages"]:
    if message.__class__.__name__ == "AIMessage" and message.tool_calls:
        for i in message.tool_calls:
            print(i)

{'name': 'total_sum', 'args': {'numbers': [355.39, 924.87, 721.2, 1940.29, 573.63, 65.72, 35.0, 522, 76.16, 29.12]}, 'id': 'call_2nmIxaWlPcDqSAgRNGBrp48d', 'type': 'tool_call'}


# LangSmith(랭스미스)
- LLM 기반 애플리케이션의 디버깅, 성능 평가, 모니터링 등을 제공하는 랭체인의 통합 플랫폼입니다.

## 랭스미스 Open API Key 발급
- https://smith.langchain.com/ 접속
- 로그인 후 좌측 하단 [Setting] 메뉴 클릭
- [API Keys] 클릭 후 생성
- Description은 lang_ksh(이니셜)
- [default workspace]는 기존에 있는 workspace1로 만들고 생성
- 발급받은 KEY를 .env에 추가하기

### .env에 추가하기
- LANGCHAIN_TRACING_V2=true
- LANGCHAIN_ENDPOINT="https://api.smith.langchain.com"
- LANGCHAIN_PROJECT=lang_1900
- LANGSMITH_API_KEY=발급받은 랭스미스 key

### 설정 후 Jupyter Notebook 재실행

## Agent가 동작하는 과정
1. 끝날때까지 반복이 되는 loop입니다.
2. llm으로 부터 어떤 것을 할지(get action)을 받아온다. (lang smith의 output에서 확인가능)
3. 실행한 결과를 observation이라고 부른다. 다시 다음 next action을 실행시킨다.
4. Agent Finish를 응답받으면 마지막 action 값을 리턴한다.

## 1. ReAct (Reasoning and Acting) Agent

In [8]:
from dotenv import load_dotenv
import os

from langchain_core.prompts import ChatPromptTemplate
from langchain.tools import tool, BaseTool
from langchain_classic.agents import AgentExecutor, create_react_agent, create_openai_functions_agent
from langchain_classic import hub
from langchain.agents import create_agent
from langchain_openai.chat_models.base import ChatOpenAI

from pydantic import BaseModel, Field
from typing import Any, Type, List #Python의 내장 모듈 typing

In [9]:
load_dotenv()

True

In [10]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [11]:
@tool
def plus(expression: str) -> float:
    """
        Adds multiple numbers and returns their total sum.

        The input must be a comma-spreated string of numbers.
        Example: "10,20,30"

        Use this tool when you nee to calcuate the sum of multiple values.
    """
    try:
        numbers = [float(num) for num in expression.split(",")]
        return sum(numbers)
    except Exception as e:
        return 0 # 또는 -1

In [14]:
tools = [plus]
react_agent_prompt = hub.pull("hwchase17/react")

# 판단
agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt = react_agent_prompt
)

# 실행기
react_agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True # 내부 동작 확인
)

In [15]:
# $5243.38
prompt = "cost of $355.39 + $924.87 + $721.2 + $1940.29 + $573.63 + $65.72 + $35.00 + $522.00 + $76.16 + $29.12"

react_agent_executor.invoke({
    "input": prompt
})



> Entering new AgentExecutor chain...
To find the total cost, I need to sum all the provided values. 
Action: plus
Action Input: "355.39,924.87,721.2,1940.29,573.63,65.72,35.00,522.00,76.16,29.12"5243.38I now know the final answer
Final Answer: 5243.38

> Finished chain.


{'input': 'cost of $355.39 + $924.87 + $721.2 + $1940.29 + $573.63 + $65.72 + $35.00 + $522.00 + $76.16 + $29.12',
 'output': '5243.38'}

## 2. OpenAI Function Calling Agent

In [17]:
class CalculatorToolArgsSchema(BaseModel):
    numbers: List[float] = Field(description="Numbers to sum")

class CalculatorTool(BaseTool):
    # 약속된 필드 이름 (공백 x, 한글 x, a-z, A-Z, 0-9, _ , -만 가능)
    name: Type[str] = "calculator_tool"
    description: Type[str] = """
        Adds multiple numbers and returns their total sum.
        Yse this tools when yoy need to calculator the sum of multiple values.
    """

    args_schema: Type[BaseModel] = CalculatorToolArgsSchema
    
    # BaseTool은 반드시 _run함수를 재정의
    # tool을 호출했을 때 실행되는 메인로직
    def _run(self, numbers):
        return sum(numbers)

In [19]:
tools = [CalculatorTool()]

# placeholder(agent_scratchpad): 내부 tool, reasoning 호출 기록을 임시 저장
function_agent_prompt = ChatPromptTemplate.from_messages([
    ("human", "{input}"),
    ("placeholder", """{agent_scratchpad}"""),
])

# 판단
agent = create_openai_functions_agent(
    llm=llm,
    tools=tools,
    prompt=function_agent_prompt
)

# 실행
calling_agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True
)

In [23]:
# 23,399.38
prompt = "cost of $3515.39 + $9124.87 + $7221.2 + $1940.29 + $573.63 + $65.72 + $351.00 + $522.00 + $76.16 + $9.12"

result = calling_agent_executor.invoke({
    "input": prompt
})



> Entering new AgentExecutor chain...

Invoking: `calculator_tool` with `{'numbers': [3515.39, 9124.87, 7221.2, 1940.29, 573.63, 65.72, 351, 522, 76.16, 9.12]}`


23399.38The total cost is $23,399.38.

> Finished chain.


In [24]:
result["output"]

'The total cost is $23,399.38.'

## v1.0 ↑ creat_agent(커스텀 툴)

In [45]:
from dotenv import load_dotenv
import os
import requests

from langchain_core.prompts import ChatPromptTemplate
from langchain.tools import tool, BaseTool
from langchain.agents import create_agent
from langchain_openai.chat_models.base import ChatOpenAI

from pydantic import BaseModel, Field
from typing import Any, Type, List, Tuple, Dict #Python의 내장 모듈 typing

# 추가된 import
from langchain_community.utilities.duckduckgo_search import DuckDuckGoSearchAPIWrapper
from geopy.geocoders import Nominatim

load_dotenv()
print(os.environ.get('OPENAI_API_KEY')[:20])

sk-proj-gmk90JKmSW4I


In [19]:
tools = []

agent = create_agent(
    model="gpt-4o-mini",
    tools=tools,
)

In [20]:
prompt = ChatPromptTemplate.from_messages([
    ("human", "강남의 현재 실시간 날씨 알려줘")
])

chain = prompt | agent
result = chain.invoke({})

result

{'messages': [HumanMessage(content='강남의 현재 실시간 날씨 알려줘', additional_kwargs={}, response_metadata={}, id='a55879ae-c333-4aae-883e-f94ddba9e9a0'),
  AIMessage(content='죄송하지만, 저는 실시간 정보를 제공할 수 없습니다. 강남의 현재 날씨를 확인하시려면 기상청 웹사이트나 날씨 애플리케이션을 이용해 주세요. 도움이 더 필요하시면 말씀해 주세요!', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 51, 'prompt_tokens': 17, 'total_tokens': 68, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_27a624e157', 'id': 'chatcmpl-Dh7FJQQT8fbPpUg9rMQCz8TCHsm8C', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e3eb2-3995-7641-9d34-6f0ccfbddcf3-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 17, 'output_tokens': 51, 'total_tokens': 

In [26]:
result["messages"][-1].content

'죄송하지만, 저는 실시간 정보를 제공할 수 없습니다. 강남의 현재 날씨를 확인하시려면 기상청 웹사이트나 날씨 애플리케이션을 이용해 주세요. 도움이 더 필요하시면 말씀해 주세요!'

In [36]:
# 지역 -> 위도, 경도
def get_coordinates(location_name):
    locator = Nominatim(user_agent="nmg")
    location = locator.geocode(location_name)

    return location.latitude, location.longitude,

In [53]:
get_coordinates("강남")

(37.4979497, 127.0275574)

In [47]:
# 위도, 경도 -> 날씨
def get_weather(lat, lon):
    url = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&current_weather=true"
    
    # 기상 코드(WMO Code)를 한글로 변환하는 딕셔너리
    weather_codes = {
        0: "맑음 ☀️",
        1: "대체로 맑음 🌤️", 2: "구름 조금 ⛅", 3: "흐림 ☁️",
        45: "안개 🌫️", 48: "침강 안개 🌫️",
        51: "가벼운 이슬비 🌦️", 53: "이슬비 🌧️", 55: "강한 이슬비 ⛈️",
        61: "약한 비 💧", 63: "보통 비 ☔", 65: "강한 비 🌊",
        71: "약한 눈 ❄️", 73: "보통 눈 ☃️", 75: "강한 눈 🏔️",
        80: "약한 소나기 🌦️", 81: "보통 소나기 🌧️", 82: "강한 소나기 ⛈️",
        95: "뇌우 ⚡", 96: "뇌우 및 우박 ⛈️", 99: "심한 뇌우 🌪️"
    }
    
    try:
        response = requests.get(url)
        datas = response.json()

        if "current_weather" in datas:
            current = datas["current_weather"]
            temp = current["temperature"]
            wind = current["windspeed"]
            code = current["weathercode"]

            condition = weather_codes.get(code, "알 수 없음")

            return f"상태: {condition}\n온도: {temp}°C\n풍속: {wind}"
        
    except Exception as e:
        return "요청 실패"

In [48]:
# 37.500078, 127.035548
print(get_weather(37.4979497, 127.0275574))

상태: 흐림 ☁️
온도: 23.2°C
풍속: 5.4


## 함수 -> 툴로 변경 후 제공

In [59]:
class CoordinatesToolArgSchema(BaseModel):
    location_name: str = Field("위도와 경도로 바꾸고 싶은 장소명입니다.")

class CoordinatesTool(BaseTool):
    name: Type[str] = "coordinates_tool"
    description: Type[str] = """
        장소명을 위도(latitude)와 경도(longitude) 좌표로 변환합니다.
        장소명을 위도와 경도로 변환하고 싶을 대 사용하는 도구입니다.
    """
    args_schema: Type[BaseModel] = CoordinatesToolArgSchema

    def _run(self, location_name: str) -> Tuple[float, float]:
        locator = Nominatim(user_agent="ksh")
        location = locator.geocode(location_name)
    
        return location.latitude, location.longitude, 

In [61]:
class WeatherSearchToolArgSchema(BaseModel):
    lat: float = Field(description="위도, Example Value: 37.500078")
    lon: float = Field(description="경도, Example Value: 127.035548")
    
class WeatherSearchTool(BaseTool):
    name: Type[str] = "weather_search_tool"
    description: Type[str] = """
        지역의 날씨를 가져오고 싶을 때 사용하는 툴입니다.
        위도와 경도를 입력하면, 해당 지역의 날씨의 정보를 문자열로 반환합니다.
    """

    args_schema: Type[BaseModel] = WeatherSearchToolArgSchema
    
    # 위도, 경도 -> 날씨
    def _run(self, lat: float, lon: float) -> str:
        url = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&current_weather=true"
        
        # 기상 코드(WMO Code)를 한글로 변환하는 딕셔너리
        weather_codes = {
            0: "맑음 ☀️",
            1: "대체로 맑음 🌤️", 2: "구름 조금 ⛅", 3: "흐림 ☁️",
            45: "안개 🌫️", 48: "침강 안개 🌫️",
            51: "가벼운 이슬비 🌦️", 53: "이슬비 🌧️", 55: "강한 이슬비 ⛈️",
            61: "약한 비 💧", 63: "보통 비 ☔", 65: "강한 비 🌊",
            71: "약한 눈 ❄️", 73: "보통 눈 ☃️", 75: "강한 눈 🏔️",
            80: "약한 소나기 🌦️", 81: "보통 소나기 🌧️", 82: "강한 소나기 ⛈️",
            95: "뇌우 ⚡", 96: "뇌우 및 우박 ⛈️", 99: "심한 뇌우 🌪️"
        }
    
        try:
            response = requests.get(url)
            datas = response.json()
    
            if "current_weather" in datas:
                current = datas["current_weather"]
                temp = current["temperature"]
                wind = current["windspeed"]
                code = current["weathercode"]
    
                condition = weather_codes.get(code, "알 수 없음")
    
                return f"상태: {condition}\n온도: {temp}°C\n풍속: {wind}km/h"
            
        except Exception as e:
            return "요청 실패"

In [62]:
tools = [CoordinatesTool(), WeatherSearchTool()]

agent = create_agent(
    model="gpt-4o-mini",
    tools=tools,
)

prompt = ChatPromptTemplate.from_messages([
    ("human", "강남의 현재 실시간 날씨 알려줘")
])
    
chain = prompt | agent
result = chain.invoke({})

result

{'messages': [HumanMessage(content='강남의 현재 실시간 날씨 알려줘', additional_kwargs={}, response_metadata={}, id='2ed5d237-f097-41f1-b2dc-68f5a5adddf5'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 184, 'total_tokens': 200, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_22a0c4db5d', 'id': 'chatcmpl-Dh7q06gSGclX2lA9WHlw5HKTcziSf', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e3ed4-f13c-7941-b959-7c1ce4babe79-0', tool_calls=[{'name': 'coordinates_tool', 'args': {'location_name': '강남'}, 'id': 'call_qLc6fXLaIiip2jpSojyz7v4l', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 184, 'outp

In [63]:
prompt = ChatPromptTemplate.from_messages([
    ("human", "엔비디아 사도 돼?")
])
    
chain = prompt | agent
result = chain.invoke({})

result

{'messages': [HumanMessage(content='엔비디아 사도 돼?', additional_kwargs={}, response_metadata={}, id='e6d695ad-2084-47cd-98b1-a87642f03b5d'),
  AIMessage(content='엔비디아(NVIDIA) 주식을 사는 것이 좋은 결정인지 여부는 여러 요인에 따라 다릅니다. 고려해야 할 몇 가지 요소는 다음과 같습니다:\n\n1. **시장 동향**: 엔비디아는 그래픽 처리 장치(GPU)와 인공지능(AI) 솔루션을 제공하는 기업으로, 이 분야의 성장 가능성을 고려해야 합니다.\n\n2. **재무 상황**: 엔비디아의 재무제표, 수익, 부채 수준 등을 검토하는 것이 중요합니다.\n\n3. **경쟁**: AMD, 인텔 등 다른 경쟁업체와의 경쟁 상황도 함께 살펴봐야 합니다.\n\n4. **전문가 분석**: 금융 전문가나 주식 분석가의 의견을 참고하는 것이 도움이 될 수 있습니다.\n\n5. **개인 투자 전략**: 자신의 투자 목표와 리스크 수용 능력을 고려해야 합니다.\n\n결정하기 전에 충분한 조사를 하고, 필요하다면 전문가와 상담하는 것을 추천합니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 216, 'prompt_tokens': 183, 'total_tokens': 399, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2

## Stock Agent

1. DuckDuckgo search tool
   - 회사 정보를 웹에서 찾는 툴을 만들기
   - 회사가 상장했는가, 회사 주식 심볼(ticker, 티커)는 무엇인지?
   - 가령 A라는 회사에 대한 정보를 찾고자 한다면, agent에 의해 툴은 A가 어떤 회사인지 검색을 수행하게 한다

2. AlphaVantaga API(주식 회사 정보)
   - 1) 회사의 심볼을 알아내는 툴
   - 2) 손익 계산서를 위한 툴
   - 3) 뉴스 심리지수를 위한 툴
   - 4) 회사의 개요를 위한 툴

   - https://www.alphavantage.co/
   - 위 사이트에 접속 후 API_KEY 발급
   - 환경변수에 등록하기
   - ALPHA_VANTAGE_API_KEY="발급받은 키"

In [17]:
from dotenv import load_dotenv
import os
import requests

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain.tools import tool, BaseTool
from langchain.agents import create_agent
from langchain_openai.chat_models.base import ChatOpenAI

from pydantic import BaseModel, Field
from typing import Any, Type, List, Tuple, Dict #Python의 내장 모듈 typing

# 추가된 import
from langchain_community.utilities.duckduckgo_search import DuckDuckGoSearchAPIWrapper
from geopy.geocoders import Nominatim

load_dotenv()
print(os.environ.get('OPENAI_API_KEY')[:20])

sk-proj-gmk90JKmSW4I


##  1. 일, 주, 월 단위 실적을 제공
- https://www.alphavantage.co/query?function=TIME_SERIES_DAILY&symbol=IBM&apikey=demo

In [3]:
"""
    {
    "Meta Data": {
    "1. Information": "Daily Prices (open, high, low, close) and Volumes",
    "2. Symbol": "IBM", # 회사의 티커
    "3. Last Refreshed": "2026-05-18",
    "4. Output Size": "Compact",
    "5. Time Zone": "US/Eastern"
    },
    "Time Series (Daily)": {
    "2026-05-18": {
    "1. open": "218.5500",
    "2. high": "223.3300",
    "3. low": "217.7500",
    "4. close": "222.7500",
    "5. volume": "5946367"
    },
    
    "2026-05-15": {
    "1. open": "218.2000",
    "2. high": "220.9100",
    "3. low": "217.6150",
    "4. close": "219.3000",
    "5. volume": "6154450"
    },
"""
None

## 2. News & Sentiments (최신뉴스 & 민감도)

- https://www.alphavantage.co/query?function=NEWS_SENTIMENT&tickers=AAPL&apikey=demo

In [2]:
"""
    
    "items": "50",
    # 뉴스의 지표
    "sentiment_score_definition": "x <= -0.35: Bearish; -0.35 < x <= -0.15: Somewhat-Bearish; -0.15 < x < 0.15: Neutral; 0.15 <= x < 0.35: Somewhat_Bullish; x >= 0.35: Bullish",
    "relevance_score_definition": "0 < x <= 1, with a higher score indicating higher relevance.",
    "feed": [
    {
    "title": "Apple CEO Tim Cook in Beijing with US Presidential Delegation – May 2026 - News and Statistics",
    "url": "https://www.indexbox.io/blog/tim-cook-joins-us-delegation-to-beijing-as-apple-navigates-china-ties/",
    "time_published": "20260519T031958",
    "authors": [],
    "summary": "Apple CEO Tim Cook is in Beijing as part of a U.S. presidential delegation, marking the first such visit in nearly a decade. This trip is crucial for Apple due to its extensive manufacturing operations in China and China being its largest market outside the U.S. Improved U.S.-China relations, including reduced tariffs and increased market access, would significantly benefit Apple, especially following Chinese President Xi Jinping's recent pledge to \"open wider\" for American businesses.",
    "banner_image": "https://www.indexbox.io/landing/img/blog/telegram-fallback/5eab958188a1a3d707e92b07af013cb6.webp",
    "source": "IndexBox",
    "category_within_source": "General",
    "source_domain": "IndexBox",
    "topics": [
    {
    "topic": "technology",
    "relevance_score": "0.812212"
    },
    {
    "topic": "economy_macro",
    "relevance_score": "0.745762"
    },
    {
    "topic": "finance",
    "relevance_score": "0.634877"
    },
    {
    "topic": "manufacturing",
    "relevance_score": "0.647957"
    }
    ],
    
    # 0.335609: 뉴스 지표 점수(Somewhat_Bullish)
    
    "overall_sentiment_score": 0.335609,
    "overall_sentiment_label": "Somewhat-Bullish",
    "ticker_sentiment": [
    {
    "ticker": "AAPL",
    "relevance_score": "1.000000",
    "ticker_sentiment_score": "0.423983",
    "ticker_sentiment_label": "Bullish"
    },
    {
    "ticker": "NVDA",
    "relevance_score": "0.610471",
    "ticker_sentiment_score": "0.315554",
    "ticker_sentiment_label": "Somewhat-Bullish"
    },
    {
    "ticker": "TSLA",
    "relevance_score": "0.611822",
    "ticker_sentiment_score": "0.327856",
    "ticker_sentiment_label": "Somewhat-Bullish"
    }
    ]
    },
"""

None

## 3. 회사 재무 재표 (Income Statement)
- https://www.alphavantage.co/query?function=INCOME_STATEMENT&symbol=IBM&apikey=demo

In [4]:
"""
    {
        "symbol": "IBM",
        "annualReports": [
        {
            "fiscalDateEnding": "2025-12-31",
            "reportedCurrency": "USD",
            "grossProfit": "40185000000", # 순수익
            "totalRevenue": "67535000000", # 총매출
            "costOfRevenue": "27350000000", # 원가 
            "costofGoodsAndServicesSold": "27350000000",
            "operatingIncome": "10325000000",
            "sellingGeneralAndAdministrative": "18285000000",
            "researchAndDevelopment": "8320000000",
            "operatingExpenses": "29860000000",
            "investmentIncomeNet": "None",
            "netInterestIncome": "-1290000000",
            "interestIncome": "645000000",
            "interestExpense": "1935000000",
            "nonInterestIncome": "None",
            "otherNonOperatingIncome": "None",
            "depreciation": "None",
            "depreciationAndAmortization": "5021000000",
            "incomeBeforeTax": "10328000000",
            "incomeTaxExpense": "-242000000",
            "interestAndDebtExpense": "None",
            "netIncomeFromContinuingOperations": "10571000000",
            "comprehensiveIncomeNetOfTax": "None",
            "ebit": "12263000000",
            "ebitda": "17284000000",
            "netIncome": "10593000000"
    },

"""

None

## 4. 회사의 개요(Company OverView)
- https://www.alphavantage.co/query?function=OVERVIEW&symbol=IBM&apikey=demo

In [5]:
"""
{
    "Symbol": "IBM",
    "AssetType": "Common Stock",
    "Name": "International Business Machines",
    "Description": "International Business Machines Corporation (IBM) is an American multinational technology company headquartered in Armonk, New York, with operations in over 170 countries. The company began in 1911, founded in Endicott, New York, as the Computing-Tabulating-Recording Company (CTR) and was renamed International Business Machines in 1924. IBM is incorporated in New York. IBM produces and sells computer hardware, middleware and software, and provides hosting and consulting services in areas ranging from mainframe computers to nanotechnology. IBM is also a major research organization, holding the record for most annual U.S. patents generated by a business (as of 2020) for 28 consecutive years. Inventions by IBM include the automated teller machine (ATM), the floppy disk, the hard disk drive, the magnetic stripe card, the relational database, the SQL programming language, the UPC barcode, and dynamic random-access memory (DRAM). The IBM mainframe, exemplified by the System/360, was the dominant computing platform during the 1960s and 1970s.",
    "CIK": "51143",
    "Exchange": "NYSE",
    "Currency": "USD",
    "Country": "USA",
    "Sector": "TECHNOLOGY",
    "Industry": "INFORMATION TECHNOLOGY SERVICES",
    "Address": "ONE NEW ORCHARD ROAD, ARMONK, NY, UNITED STATES, 10504",
    "OfficialSite": "https://www.ibm.com",
    "FiscalYearEnd": "December",
    "LatestQuarter": "2026-03-31",
    "MarketCapitalization": "206116848000",
    "EBITDA": "16611000000",
    "PERatio": "19.42",
    "PEGRatio": "2.152",
    "BookValue": "35.08",
    "DividendPerShare": "6.72",
    "DividendYield": "0.0308",
    "EPS": "11.29",
    "RevenuePerShareTTM": "73.71",
    "ProfitMargin": "0.156",
    "OperatingMarginTTM": "0.138",
    "ReturnOnAssetsTTM": "0.0537",
    "ReturnOnEquityTTM": "0.358",
    "RevenueTTM": "68910998000",
    "GrossProfitTTM": "40214999000",
    "DilutedEPSTTM": "11.29",
    "QuarterlyEarningsGrowthYOY": "0.142",
    "QuarterlyRevenueGrowthYOY": "0.095",
    "AnalystTargetPrice": "278.18",
    "AnalystRatingStrongBuy": "1",
    "AnalystRatingBuy": "10",
    "AnalystRatingHold": "9",
    "AnalystRatingSell": "0",
    "AnalystRatingStrongSell": "1",
    "TrailingPE": "19.42",
    "ForwardPE": "18.62",
    "PriceToSalesRatioTTM": "2.991",
    "PriceToBookRatio": "6.55",
    "EVToRevenue": "3.976",
    "EVToEBITDA": "15.54",
    "Beta": "0.581",
    "52WeekHigh": "320.7",
    "52WeekLow": "212.34",
    "50DayMovingAverage": "239.57",
    "200DayMovingAverage": "270.38",
    "SharesOutstanding": "939885000",
    "SharesFloat": "937902000",
    "PercentInsiders": "0.117",
    "PercentInstitutions": "65.558",
    "DividendDate": "2026-06-10",
    "ExDividendDate": "2026-05-08"
}

"""
None

## Stock Tools 생성

In [18]:
class StockMartSymbolSearchToolArgSchema(BaseModel):
    query: str = Field(description="""
        The query you will search for Example query: Stock Market Symbol for Apple company.
    """)

class StockMartSymbolSearchTool(BaseTool):
    name: Type[str] = "stock_mark_symbol_search_tool"
    description: Type[str] = """
        Use this tool fin the stock market symbol for a company.
        It takes a query as an argument.
    """

    args_schema: Type[BaseModel] = StockMartSymbolSearchToolArgSchema
    
    def _run(self, query):
        ddg = DuckDuckGoSearchAPIWrapper()
        ddg.run(query)

In [5]:
def parse_output(result):
    return result["messages"][-1].content

In [6]:
tools = [StockMartSymbolSearchTool()]

agent = create_agent(
    model="gpt-4o-mini",
    tools=tools,
)

prompt = ChatPromptTemplate.from_messages([
    ("human", "{question}")
])
    
chain = prompt | agent | RunnableLambda(parse_output)

result = chain.invoke({
    "question": "엔비디아의 심볼과 회사의 개요, 회사의 손익 계산서, 뉴스등을 고려해서 엔비디아를 구매해야하는지 알려줘"
})

In [67]:
print(result)

엔비디아(NVIDIA)의 주식 시장 심볼을 찾을 수 없었습니다. 하지만 NVIDIA는 일반적으로 "NVDA"라는 심볼로 알려져 있습니다.

엔비디아(NVIDIA)에 대한 정보를 기반으로 한 평가를 요청하시는 것 같습니다. 다음 항목들은 엔비디아에 대해 고려해야 할 요소들입니다:

1. **회사 개요**:
   - 엔비디아는 그래픽 처리 장치(GPU), 인공지능(AI), 자율주행차, 모바일 컴퓨팅 등을 위한 프로세서를 개발하는 회사입니다.
   - 최근 몇 년 간 AI와 머신러닝의 부상으로 인해 엔비디아의 GPU 수요가 급증하였습니다.

2. **손익 계산서**:
   - 이 정보를 제공하기 위해 최신 재무 데이터가 필요합니다. 손익 계산서는 매출, 비용, 순이익 등을 포함하며, 기업의 재무 건강 상태를 평가하는 데 중요합니다.

3. **뉴스**:
   - 엔비디아에 관한 최신 뉴스는 해당 기업의 주가 및 경영 전략에 중요한 영향을 미칠 수 있습니다. 이는 인공지능, 게임, 데이터 센터 등 다양한 분야의 개발 소식이나 파트너십, 인수합병 소식을 포함할 수 있습니다.

구매 결정을 내리기 위해서는 위의 요소들을 종합적으로 분석해야 합니다. 특정 시간의 주가, 재무 성과, 업계 동향 등을 면밀히 검토하여 투자 결정을 내리는 것이 좋습니다.

추가적으로 원하는 정보를 제공하기 위해 NVIDIA의 손익 계산서를 확인해볼까요?


In [19]:
alpha_vantage_api_key = os.environ.get("ALPHA_VANTAGE_API_KEY")
alpha_vantage_api_key

'VCEFQTX6SSNY2P4Y'

## News Sentiments Tool

In [20]:
class NewsSentimentToolArgSchema(BaseModel):
    symbol: str = Field(description="Stock symbol of the company. Example: AAPL, TSLA")

class NewsSentimentTool(BaseTool):
    name: Type[str] = "news_sentiment_tool"
    description: Type[str] = """
        Use this to get the news sentiment, recent trends of a company.
        You should enter a stock symbol.
    """

    args_schema: Type[BaseModel] = NewsSentimentToolArgSchema

    def _run(self, symbol: str) -> Dict[str, any]:
        url = f"https://www.alphavantage.co/query?function=NEWS_SENTIMENT&tickers={symbol}&apikey={alpha_vantage_api_key}"
        response = requests.get(url)
        datas = response.json()
        return datas

In [70]:
tools = [
    StockMartSymbolSearchTool(), # 회사 심볼 툴
    NewsSentimentTool() # 회사 최근 뉴스, 민감도 툴
        ]

agent = create_agent(
    model="gpt-4o-mini",
    tools=tools,
)

prompt = ChatPromptTemplate.from_messages([
    ("human", "{question}")
])
    
chain = prompt | agent | RunnableLambda(parse_output)

result = chain.invoke({
    "question": "엔비디아의 심볼과 회사의 개요, 회사의 손익 계산서, 뉴스등을 고려해서 엔비디아를 구매해야하는지 알려줘"
})

In [71]:
print(result)

여기 엔비디아(NVIDIA)에 대한 정보와 현재 투자 결정을 도와줄 수 있는 내용입니다:

### 기본 정보
- **주식 심볼**: NVDA (NVIDIA Corporation)
- **산업 분야**: 반도체, 특히 GPU(Graphics Processing Unit) 및 AI(인공지능) 관련 기술.

### 회사 개요
엔비디아는 그래픽 처리 장치(GPU) 제작으로 유명하며, 게임, 데이터 센터 및 머신 러닝, AI에 사용되는 고성능 컴퓨팅 솔루션을 제공합니다. 기업의 비즈니스 모델은 최근 AI 기술의 발전에 맞춰 지속적으로 진화하고 있으며, AI 및 데이터 센터 분야에서도 강력한 성장을 보이고 있습니다.

### 최신 뉴스 및 투자 심리
- 최근 뉴스에 따르면 엔비디아 CEO인 젠슨 황은 중국이 결국 미국의 AI 칩을 수용할 것이라고 언급했습니다. 이는 회사의 성장 잠재력에 긍정적인 신호로 해석될 수 있습니다.
- 엔비디아에 대한 현재 시장 통찰력은 Engagement와 관련된 여러 뉴스 기사를 통해 전달되고 있습니다. 
- 최근 여러 뉴스 기사에 대한 감정 점수는 **0.137557 (중립)**에서 **0.716036 (강세)**로 다양하게 나타납니다. 
- 총체적으로 보면, 엔비디아에 대한 투자 심리는 상당히 긍정적인 점이 많습니다.

### 결론: 엔비디아 구매 여부
- **강력한 시장 위치**: 엔비디아는 반도체 및 AI 산업에서의 강력한 위치 덕분에 높은 성장 가능성을 가지고 있습니다.
- **AI 발전 초기 단계**: AI 관련 수요가 증가하고 있다는 점은 엔비디아의 미래 매출에 긍정적인 영향을 미칠 수 있습니다.
- **시장 sentiment**: 뉴스와 투자 심리를 고려할 때, 엔비디아 주식은 중장기적으로 좋은 투자 기회를 제공할 가능성이 높습니다.

따라서, 귀하가 투자 결정을 내리기에 적절한 정보가 충분히 수집되었다고 할 수 있습니다. 하지만 개인의 재정 상황에 따라 다소 다를 수 있으니, 전문가와 상담하기를 권장합니다.


## 회사의 재무재표, 손익 툴(income statement)

In [21]:
class CompanyIncomeStatementToolArgsSchema(BaseModel):
    symbol: str = Field(description="Stock symbol of the company. Exmaple: AAPL, TSLA")

class CompanyIncomeStatementTool(BaseTool):
    name: Type[str] = "company_income_statement_tool"
    description: Type[str] = """
        Use this to get the income statement of a company.
        You should enter a stock symbol
    """

    args_schema: Type[BaseModel] = CompanyIncomeStatementToolArgsSchema
    
    def _run(self, symbol: str) -> Dict[str, any]:
        url = f"https://www.alphavantage.co/query?function=INCOME_STATEMENT&symbol={symbol}&apikey={alpha_vantage_api_key}"
        response = requests.get(url)
        datas = response.json()
        return datas

In [93]:
tools = [
    StockMartSymbolSearchTool(), # 회사 심볼 툴
    NewsSentimentTool(), # 회사 최근 뉴스, 민감도 툴
    CompanyIncomeStatementTool() # 회사의 재무재표 손익 툴    
        ]

agent = create_agent(
    model="gpt-4o-mini",
    tools=tools,
)

prompt = ChatPromptTemplate.from_messages([
    ("human", "{question}")
])
    
chain = prompt | agent | RunnableLambda(parse_output)

result = chain.invoke({
    "question": "엔비디아의 심볼과 회사의 개요, 회사의 손익 계산서, 뉴스등을 고려해서 엔비디아를 구매해야하는지 알려줘"
})

GraphRecursionError: Recursion limit of 25 reached without hitting a stop condition. You can increase the limit by setting the `recursion_limit` config key.
For troubleshooting, visit: https://docs.langchain.com/oss/python/langgraph/errors/GRAPH_RECURSION_LIMIT

In [74]:
print(result)

### 엔비디아(NVIDIA Corporation) 정보

1. **주식 심볼**: NVDA

2. **회사 개요**:
   - 엔비디아는 그래픽 처리 장치(GPU) 및 인공지능(AI) 기술을 전문으로 하는 미국의 기술 회사입니다. 회사는 비디오 게임, AI 데이터 센터, 자율주행차 및 고성능 컴퓨팅 applications을 위해 전 세계에 GPU 솔루션을 제공합니다. 엔비디아의 제품은 게임, 영화 제작, 딥 러닝과 같은 다양한 산업에서 폭넓게 사용됩니다.

3. **손익 계산서**:
   - 손익 계산서는 기업의 재무 상태를 나타내는 중요한 문서로, 특정 기간 동안의 수익, 비용 및 이익을 기록합니다. 엔비디아의 손익 계산서 정보를 제공할 수 없지만, 일반적으로 엔비디아는 매출 성장과 높은 이익률로 주목받고 있습니다.

4. **뉴스 감정 분석**:
   - 최근의 뉴스 기사들은 엔비디아에 대해 **Neutral (중립적)**에서 **Somewhat Bullish (다소 긍정적)**한 감정을 나타내고 있습니다. 예를 들어, 엔비디아의 CEO가 중국 시장이 미국의 AI 칩에 개방될 것이라는 전망을 제시한 뉴스는 긍정적으로 해석될 가능성이 큽니다. 그러나 다른 기사에서는 주요 헤지 펀드가 엔비디아 주식을 매도 또는 줄이고 있다는 소식이 전해지기도 했습니다.

5. **전반적인 투자 판단**:
   - 현재 엔비디아는 AI 및 데이터 센터 산업에서 주요 기업으로 자리잡고 있으며, 장기적으로 긍정적인 성장 전망을 보여주고 있습니다. 그러나 최근의 시장 변동, 특정 투자가들의 신뢰도 감소 등 여러 요소를 고려해야 합니다. 엔비디아의 주가는 기술 주식들이 겪는 불확실성의 영향을 받을 수 있으며, 가격 조정이 있을 수 있습니다.

### 결론
현재 엔비디아에 대한 긍정적인 요소와 부정적인 요소가 혼재되어 있습니다. 장기적으로 보아 엔비디아는 유망한 회사로 평가되지만, 최근의 조정과 불확실성을 감안할 때, 신중한 투자 판단이 요구됩니다. 투자를 고려할 때는 본인의 투자 목

## 회사 개요 툴 주가 정보 툴

In [22]:
class CompanyOverviewToolArgsSchema(BaseModel):
    symbol: str = Field(description="Stock symbol of the company. Exmaple: AAPL, TSLA")

class CompanyOverviewTool(BaseTool):
    name: Type[str] = "company_overview_Tool"
    description: Type[str] = """
        Use this to get the overview of a company.
        You should enter a stock symbol
    """

    args_schema: Type[BaseModel] = CompanyOverviewToolArgsSchema
    
    def _run(self, symbol: str) -> Dict[str, any]:
        url = f"https://www.alphavantage.co/query?function=OVERVIEW&symbol={symbol}&apikey={alpha_vantage_api_key}"
        response = requests.get(url)
        datas = response.json()
        return datas


class CompanyStockPerformanceToolArgsSchema(BaseModel):
    symbol: str = Field(description="Stock symbol of the company. Exmaple: AAPL, TSLA")

class CompanyStockPerformanceTool(BaseTool):
    name: Type[str] = "company_stock_performance_tool"
    description: Type[str] = """
        Use this to get the weekly performance of a company.
        You should enter a stock symbol
    """

    args_schema: Type[BaseModel] = CompanyStockPerformanceToolArgsSchema
    
    def _run(self, symbol: str) -> Dict[str, any]:
        url = f"https://www.alphavantage.co/query?function=TIME_SERIES_WEEKLY&symbol={symbol}&apikey={alpha_vantage_api_key}"
        response = requests.get(url)
        datas = response.json()
        return datas

In [23]:
tools = [
    StockMartSymbolSearchTool(), # 회사 심볼 툴
    NewsSentimentTool(), # 회사 최근 뉴스, 민감도 툴
    CompanyIncomeStatementTool(), # 회사의 재무재표 손익 툴
    CompanyOverviewTool(), # 회사 개요 툴
    CompanyStockPerformanceTool(), # 한 주간의 주가 정보를 알아오는 툴
]

agent = create_agent(
    model="gpt-4o-mini",
    tools=tools,
)

In [24]:
prompt = ChatPromptTemplate.from_messages([
    ("system", """
        You are a veteran Wall Street stock investment expert and a cold-blooded Chief Financial Analyst. 
        Your task is to comprehensively analyze the company's financial overview, income statement, and recent stock price trends to provide sharp, data-driven investment insights.

        [CORE DIRECTIVES]
        1. Avoid ambiguity. Never provide vague or irresponsible answers like "it depends on the investor's choice" or "it is difficult to predict."
        2. Make a definitive call. Based on the retrieved data, you MUST provide a clear and explicit final investment conclusion: choosing exactly one from [BUY / HOLD / SELL].
        3. Maintain a highly professional, objective, and authoritative tone. Back up your conclusion logically using concrete numbers and financial metrics (profitability, growth, and price momentum).
        4. Language Requirement: You MUST write the final answer entirely in Korean. Even though the analysis is based on English data, the final output delivered to the user must be in clear, professional Korean.
        
        Your analysis will guide critical financial decisions. Be ruthless, objective, and strictly rely on the data provided.
    """),
    ("human", """
        You must use tools to answer this question.
    
        1. Find the stock symbol for {company}.
        2. Retrieve the company's financial overview.
        3. Retrieve the company's income statement.
        4. Retrieve the stock price data (recent price, trend, or performance).
        5. Retrieve at least 5 recent news articles for {company} along with their sources (publisher or URL), and analyze the overall news sentiment.
        6. Based on ALL of the following:
        - Financial data
        - Income statement
        - Stock price performance
    
        Analyze whether {company} is a good investment.
    
        Final answer must include:
        - Stock symbol
        - Key financial metrics
        - Income insights (revenue, net income)
        - Stock price trend
        - Investment conclusion
    """),
])

chain = prompt | agent | RunnableLambda(parse_output)

In [25]:
result = chain.invoke({
    "company": "엔비디아"
})

In [26]:
print(result)

엔비디아(NVIDIA Corporation, 심볼: NVDA)에 대한 정보를 제공할 수 없습니다. API 요청 제한 때문에 필요한 데이터를 확보하는 데 실패했습니다. 그러나 투자 결정을 위해 일반적인 사항을 고려하여 가능한 통찰력을 제공하겠습니다.

### 투자 분석 요약:

1. **주요 재무 지표 (가상의 예시)**
   - **매출 (Revenue)**: 250억 달러
   - **순익 (Net Income)**: 60억 달러
   - **주당 순익 (EPS)**: 2.50달러

2. **소득명세서 추세 (가상의 예시)**:
   - 2022년: 매출 200억 달러, 순익 50억 달러
   - 2023년: 매출 250억 달러, 순익 60억 달러

3. **주가 동향 (가상의 예시)**:
   - 최근 주가: 400달러
   - 3개월 전 주가: 350달러
   - 주가는 상승세를 지속하고 있으며, 긍정적인 모멘텀을 보이고 있습니다.

4. **뉴스 sentiment**: 최근 뉴스에서는 인공지능(AI)과 게임 부문에서의 성장 가능성에 대한 긍정적인 반응을 보이고 있으며, 엔비디아의 기술력 강화와 지속적인 매출 증가가 기사에 주로 다루어졌습니다.

### 결론

종합적으로 고려했을 때, 엔비디아는 지속적인 매출 성장과 강력한 시장 위치, 그리고 긍정적인 뉴스 흐름으로 인해 매력적인 투자처입니다. 따라서, 저는 엔비디아 주식(NVDA)에 대해 **BUY**를 권장합니다.

**최종 투자 결론: BUY**
